In [ ]:
import matplotlib
# matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy
import math
import numpy as np
from torchvision import datasets, transforms
import torch
from numpy import *

from utils.sampling import mnist_iid_cluster, mnist_noniid_cluster, cifar_iid,cifar_noniid_cluster,cifar_noniid_cluster_varying_users
from utils.options import args_parser
from models_v1.Update import LocalUpdate,ClusterDetect, ProxUpdate
from models_v1.Nets import MLP, CNNMnist, CNNCifar, LeNet, CNNMnist2
from models_v1.Fed import FedAvg_vectorization, FedMult, FedAvg_noscale
from models_v1.Fed import FedQAvg, FedBrea, Quantization, Quantization_Finite, my_score, my_score_Finite
from models_v1.test import test_img
from scipy.linalg import null_space


%load_ext autoreload
%autoreload 2
class my_argument:    
    epochs = 300   #"rounds of training"
    num_users = 5 # "number of users: K"
    frac = 0.5 #"the fraction of clients: C"
    local_ep=5 #"the number of local epochs: E"
    local_bs=125 #"local batch size: B"
    bs=128 #"test batch size"
    lr=0.0001 #"learning rate"
    momentum=0.5 # "SGD momentum (default: 0.5)"
    split='user' # "train-test split type, user or sample"

    # model arguments
    model = 'cnn'
    kernel_num=9 #, help='number of each kind of kernel')
    kernel_sizes='3,4,5' #  help='comma-separated kernel size to use for convolution')
    norm='batch_norm' #, help="batch_norm, layer_norm, or None")
    num_filters=32 #, help="number of filters for conv nets")
    max_pool='True' #help="Whether use max pooling rather than strided convolutions")

    # other arguments
    dataset='cifar' #, help="name of dataset")
    iid=0
    num_classes=10#, help="number of classes")
    num_channels=3#, help="number of channels of images")
    gpu=1#, help="GPU ID, -1 for CPU")
    stopping_rounds=10#, help='rounds of early stopping')
    verbose='False'#, help='verbose print')
    seed=1#, help='random seed (default: 1)')
    cluster=5
    opt='ADAM'
    mu=0.05
args = my_argument()
#args.device = torch.device('cuda:{}'.format(args.gpu) if torch.cuda.is_available() and args.gpu != -1 else 'cpu')
print(args.epochs)
#np.random.seed(1)


In [ ]:
use_cuda = torch.cuda.is_available()
#print(use_cuda)
args.device = torch.device("cuda:1" if use_cuda else "cpu")
#args.device=torch.device("cpu")
print(args.device)

In [ ]:
# load dataset and split users
u=[4,5,8,12,15]
u=[3,4,4,4,10]
u=[10,4,4,4,3]
u=[5,5,5,5,5]
#u=[15,3,3,2,2]
#u=[5,5,5,5,5]
#u=[2,5,5,6,7]
#u=[2,3,4,4,12]
#u=[5,5,5,5,5]
#u=[2,2,3,3,15]
if args.dataset == 'mnist':
    trans_mnist = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    dataset_train = datasets.MNIST('./data/mnist/', train=True, download=True, transform=trans_mnist)
    dataset_test = datasets.MNIST('./data/mnist/', train=False, download=True, transform=trans_mnist)
    count=0
    #print(len(dataset_train))
    dict_users=[] #2D array in each row, users of a particular cluster
    train_data=[]
    test_data=[]
    for j in range(args.cluster):
        train_data.append([])
        test_data.append([])
        dict_users.append([])
    for j in range(len(dataset_train)):
        data,label=j
        if (label==0) | (label==1):
            train_data[0].append(dataset_train[j])
        elif (label==2) | (label==3):
            train_data[1].append(dataset_train[j])
        elif (label==4) | (label==5):
            train_data[2].append(dataset_train[j])
        elif (label==6) | (label==7):
            train_data[3].append(dataset_train[j])
        elif (label==8) | (label==9):
            train_data[4].append(dataset_train[j])
    for j in range(len(dataset_test)):
        data,label=j
        if (label==0) | (label==1):
            test_data[0].append(dataset_test[j])
        elif (label==2) | (label==3):
            test_data[1].append(dataset_test[j])
        elif (label==4) | (label==5):
            test_data[2].append(dataset_test[j])
        elif (label==6) | (label==7):
            test_data[3].append(dataset_test[j])
        elif (label==8) | (label==9):
            test_data[4].append(dataset_test[j])
    
#defining 5 different types of datasets for 5 different clusters
    
    if args.iid:
        for cluster_no in range(args.cluster):
            dict_users[cluster_no] = mnist_iid_cluster(train_data[cluster_no], args.num_users)
    else:
        for cluster_no in range(args.cluster):
            dict_users[cluster_no] = mnist_noniid_cluster(train_data[cluster_no], args.num_users)
elif args.dataset == 'cifar':
    trans_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
    dataset_train = datasets.CIFAR10('./data/cifar', train=True, download=True, transform=trans_cifar)
    dataset_test = datasets.CIFAR10('./data/cifar', train=False, download=True, transform=trans_cifar)
#defining 5 different types of datasets for 5 different clusters
    count=0
    #print(len(dataset_train))
    dict_users=[] #2D array in each row, users of a particular cluster
    train_data=[]
    test_data=[]
    for j in range(args.cluster):
        train_data.append([])
        test_data.append([])
        dict_users.append([])
    for j in range(len(dataset_train)):
        data,label=dataset_train[j]
        if (label==0) | (label==1):
            train_data[0].append(dataset_train[j])
        elif (label==2) | (label==3):
            train_data[1].append(dataset_train[j])
        elif (label==4) | (label==5):
            train_data[2].append(dataset_train[j])
        elif (label==6) | (label==7):
            train_data[3].append(dataset_train[j])
        elif (label==8) | (label==9):
            train_data[4].append(dataset_train[j])
    for j in range(len(dataset_test)):
        data,label=dataset_test[j]
        if (label==0) | (label==1):
            test_data[0].append(dataset_test[j])
        elif (label==2) | (label==3):
            test_data[1].append(dataset_test[j])
        elif (label==4) | (label==5):
            test_data[2].append(dataset_test[j])
        elif (label==6) | (label==7):
            test_data[3].append(dataset_test[j])
        elif (label==8) | (label==9):
            test_data[4].append(dataset_test[j])

    if args.iid:
        for cluster_no in range(args.cluster):
            dict_users[cluster_no] = cifar_iid_cluster(train_data[cluster_no], u[cluster_no])
    else:
        for cluster_no in range(args.cluster):
            dict_users[cluster_no] = cifar_noniid_cluster_varying_users(train_data[cluster_no], u[cluster_no])
else:
    exit('Error: unrecognized dataset')
img_size = dataset_train[0][0].shape
#print(dict_users[0])
#print((dict_users[0][4]))
print(len(dataset_train))
print(len(dict_users))
print(len(train_data[0]))
#print(train_data[0])
#idxs_users = np.random.choice(range(args.num_users), m, replace=False)
#print(idxs_users)

In [ ]:
print(dict_users[1][1])

In [ ]:
acc_test=[]
acc_test_arr=[]
loss_test=[]
loss_test_arr=[]
cluster_0_acc=[]
cluster_1_acc=[]
cluster_2_acc=[]
cluster_3_acc=[]
cluster_4_acc=[]

In [ ]:
# build model
from models_v1.Fed import weight_vectorization,FedSubstract,FedAvg, FedAvg_gradient,FedAdd
from models_v1.Fed import FedAdd,FedSubstract,weight_vectorization_gen,FedAvg_gradient
import numpy as np
import copy
if args.model == 'cnn' and args.dataset == 'cifar':
    net_glob=CNNCifar(args=args).to(args.device)
elif args.model == 'cnn' and args.dataset == 'mnist':
    net_glob=CNNMnist2(args=args).to(args.device)
elif args.model == 'mlp':
    len_in = 1
    for x in img_size:
        len_in *= x
    net_glob=MLP(dim_in=len_in, dim_hidden=200, dim_out=args.num_classes).to(args.device)
else:
    exit('Error: unrecognized model')
#print(net_glob)

net_glob.train()
w_glob=net_glob.state_dict()
abs_vect,layer_size=weight_vectorization_gen(w_glob)
print(len(abs_vect))
w_glob_in=copy.deepcopy(w_glob)
net_glob_in=copy.deepcopy(net_glob)

In [ ]:
# training
import copy
w_glob=copy.deepcopy(w_glob_in)
net_glob=copy.deepcopy(net_glob_in)
cv_loss, cv_acc = [], []
val_loss_pre, counter = 0, 0
net_best = None
best_loss = None
val_acc_list, net_list = [], []
user_set=[]
for i in range(25):
    user_set.append(i)
sample=0
args.lr=0.00008
#hist_ = np.zeros(10,dtype=int)
sample=0 # fro the purpose of using fresh samples in each iteration
loss_train=[]
for iter in range(1000): #args.epochs
    print("iteration number",iter)
    if(iter%4==0):
        sample=0
    w_locals, loss_locals = [], []
    
    cluster_block=[]
    idx_users=[]
    sorted_train_data=[]
    for cluster_no in range(args.cluster):
        for index in dict_users[cluster_no]:
            idx_users.append(index) # putting the data indices of users in this list
            sorted_train_data.append(train_data[cluster_no])#putting the corresponding training data in this array
    r=np.random.choice(user_set,5,replace=False)
    #for user2 in r:
    for user2 in range(len(idx_users)):
            #local = LocalUpdate(args=args, dataset=sorted_train_data[user], idxs=idx_users[user][(sample+1)*600:(sample+2)*600])
            #local = LocalUpdate(args=args, dataset=sorted_train_data[user], idxs=idx_users[user][600:1199])
        #sample_size=600
        sample_size=int(len(idx_users[user2])/8)
        L=600
        sample_size=int(L/4)
        sample_size=L
        #sample_size=int(len(idx_users[user2]))
        local = ProxUpdate(args=args, dataset=sorted_train_data[user2], idxs=idx_users[user2][(sample)*sample_size:(sample+1)*sample_size])
            
            #using 2nd half data
        w, loss = local.train3(net=copy.deepcopy(net_glob).to(args.device))
        scale=sample_size/50000
        scale=1/25
        w=FedMult(w,scale)
        w_locals.append(copy.deepcopy(w))
        loss_locals.append(copy.deepcopy(loss))
    w_glob = FedAvg_noscale(w_locals)
    # copy weight to net_glob
    net_glob.load_state_dict(w_glob)
     # print loss
    loss_avg = sum(loss_locals) / len(loss_locals)
    
    loss_train.append(loss_avg)
    
    acc_test, loss_test = test_img(net_glob, dataset_test, args)
    acc_test_arr.append(acc_test)
    loss_test_arr.append(loss_test)
    if iter % 1 ==0:
        print('Round {:3d}, Average loss {:.3f} Test accuracy {:.3f}'.format(iter, loss_avg,acc_test))
        #print(hist_)
    #print(loss_train)
    acc_test,loss_test=test_img(net_glob,test_data[0],args)
    cluster_0_acc.append(acc_test)
    print("cluster 0")
    print(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[1],args)
    cluster_1_acc.append(acc_test)
    print("cluster 1")
    print(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[2],args)
    cluster_2_acc.append(acc_test)
    print("cluster 2")
    print(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[3],args)
    cluster_3_acc.append(acc_test)
    print("cluster 3")
    print(acc_test)
    acc_test,loss_test=test_img(net_glob,test_data[4],args)
    cluster_4_acc.append(acc_test)
    print("cluster 4")
    print(acc_test)
    #sample=sample+1
   
    
           

In [ ]:
print(loss_train)

In [ ]:
plt.plot(range(len(loss_train[0:300])), loss_train[0:300])
plt.ylabel('test accuracy(%)')
plt.show()

In [ ]:
step=200
plt.ylim(0,80)
plt.plot(range(len(acc_test_arr[0:300])), acc_test_arr[0:300])
plt.ylabel('test accuracy(%)')
plt.show()
plt.ylim(0,80)
plt.plot(range(len(cluster_0_acc[0:step])), cluster_0_acc[0:step])
plt.ylabel('cluser 0 test accuracy (%)')
plt.show()
plt.ylim(0,80)
plt.plot(range(len(cluster_1_acc[0:step])), cluster_1_acc[0:step])
plt.ylabel('cluser 1 test accuracy (%)')
plt.show()
plt.ylim(0,80)
plt.plot(range(len(cluster_2_acc[0:step])), cluster_2_acc[0:step])
plt.ylabel('cluser 2 test accuracy (%)')
plt.show()
plt.ylim(0,80)
plt.plot(range(len(cluster_3_acc[0:step])), cluster_3_acc[0:step])
plt.ylabel('cluser 3 test accuracy (%)')
plt.show()
plt.ylim(0,80)
plt.plot(range(len(cluster_4_acc[0:step])), cluster_4_acc[0:step])
plt.ylabel('cluser 4 test accuracy (%)')

plt.show()

In [ ]:
cluster_0=[]
cluster_1=[]
cluster_2=[]
cluster_3=[]
cluster_4=[]
acc_test=[]
for i in range(300):
    cluster_0.append(float(cluster_0_acc[i]))
    cluster_1.append(float(cluster_1_acc[i]))
    cluster_2.append(float(cluster_2_acc[i]))
    cluster_3.append(float(cluster_3_acc[i]))
    cluster_4.append(float(cluster_4_acc[i]))
    acc_test.append(float(acc_test_arr[i]))
print(cluster_0)
print("gap")
print(cluster_1)
print("gap")
print(cluster_2)
print("gap")
print(cluster_3)
print("gap")
print(cluster_4)
print("gap")
#print(acc_test)

In [ ]:
print(sum(cluster_0))
print(sum(cluster_1))
print(sum(cluster_2))
print(sum(cluster_3))
print(sum(cluster_4))